# Reproduce AERO-DETR Pipeline — Table 3 (Validation metrics per runway config)

Workspace-relative copy of `AERO_DETR_Pipeline_all.ipynb`. Functions are unchanged. Outputs go to `reproduce_output_<cat>`; existing files are not touched. Per-raster inference timing added. Runs the `inter` (intersection) class to validate first.

In [ ]:
# ── One-time setup: dependencies + dataset/models from Zenodo ─────────────────
# Google Colab: first clone the repo and cd into it (see README.md), then run this
# cell. Locally it is effectively a no-op if deps are installed and the data is
# already present (1_download_dataset.py skips existing rasters/*.jp2 & models/*.pt).
import sys, subprocess

def _sh(cmd):
    print('>', cmd)
    subprocess.run(cmd, shell=True, check=False)

# GDAL system libraries are required before the Python bindings on Colab/Linux.
try:
    from osgeo import gdal  # noqa: F401
except Exception:
    if sys.platform.startswith('linux'):
        _sh('apt-get -qq install -y gdal-bin libgdal-dev')
        _sh(f'{sys.executable} -m pip install -q "GDAL==$(gdal-config --version)"')

# Python dependencies
_sh(f'{sys.executable} -m pip install -q -r requirements.txt')

# Dataset (rasters/*.jp2) + model checkpoints (*.pt) from Zenodo — ~5 GB, one time
_sh(f'{sys.executable} 1_download_dataset.py')


In [ ]:
import os
import glob
from pathlib import Path
import ntpath
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None
from ultralytics import YOLO
import argparse
import torch
from osgeo import gdal

In [ ]:
from geo_utils import *
from extract_runway_markings import *

In [ ]:
def obb_predictions_to_geodf(src_raster, results):
    src = gdal.Open(src_raster)
    ulx, xres, xskew, uly, yskew, yres = src.GetGeoTransform()
    crs_wkt = src.GetProjection()
    ulx = float(ulx); xres = float(xres); xskew = float(xskew)
    uly = float(uly); yskew = float(yskew); yres = float(yres)
    forward_transform = (ulx, xres, xskew, uly, yskew, yres)
    del src
    result = results[0]
    detections_list = []
    if result.obb:
        for cls, conf, xywhr, obb in zip(result.obb.cls, result.obb.conf, result.obb.xywhr, result.obb.xyxyxyxy):
            pixels_x, pixels_y = map(list, zip(*obb))
            latlong_list = []
            for x, y in zip(pixels_x, pixels_y):
                pixel_coord = gdal.ApplyGeoTransform(forward_transform, float(x), float(y))
                latlong_list.append(pixel_coord)
            poly = geometry.Polygon([[lon, lat] for lon, lat in latlong_list])
            result_dict = {'class_id': int(cls.item()), 'conf': conf.item(), 'geometry': poly}
            detections_list.append(result_dict)
    df_preds = pd.DataFrame.from_dict(detections_list)
    gdf_preds = gpd.GeoDataFrame(df_preds)
    gdf_preds.set_crs(crs_wkt, inplace=True)
    return gdf_preds

In [ ]:
import rasterio
import geopandas as gpd
from shapely.geometry import box

def clip_by_raster_bounding_box(gdf, input_raster):
    with rasterio.open(input_raster) as rast_src:
        bbox = rast_src.bounds
    bbox_polygon = box(bbox.left, bbox.bottom, bbox.right, bbox.top)
    gdf = gdf.to_crs(rast_src.crs)
    bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox_polygon]}, crs=rast_src.crs)
    clipped_gdf = gpd.clip(gdf, bbox_gdf)
    return clipped_gdf

In [ ]:
import numpy as np

def predict_rwy_potential(model_obb, input_raster, out_path, gen_int=True):
    input_raster_name = Path(input_raster).stem
    analysis_folder = Path(out_path) / f"{input_raster_name}"
    analysis_folder.mkdir(parents=True, exist_ok=True)
    print(f"Created preds directory: {analysis_folder}")
    if torch.cuda.is_available():
        device = 0
    else:
        device = 'cpu'
    # Read pixels via rasterio so both JPEG2000 (.jp2, the Zenodo format) and
    # GeoTIFF (.tif) decode reliably; ultralytics' default image reader cannot
    # open .jp2. Drop any 4th (alpha) band, then RGB -> BGR for ultralytics.
    with rasterio.open(input_raster) as rsrc:
        bands = rsrc.read()                       # (C, H, W)
    rgb = np.transpose(bands[:3], (1, 2, 0))      # (H, W, 3), keep first 3 bands
    pred_src = np.ascontiguousarray(rgb[:, :, ::-1])
    results = model_obb.predict(pred_src, save=True, save_txt=True, save_conf=True, device=device, max_det=10, show_boxes=False)
    print('Completed Coarse predictions')
    obb_gdf = obb_predictions_to_geodf(input_raster, results)
    obb_gdf = retain_larger_and_non_overlapping_polygons(obb_gdf, 25)
    medial_axis_gdf = calculate_medial_axis(obb_gdf)
    extended_gdf = extend_medial_axis_to_bounds(medial_axis_gdf, input_raster)
    buffered_gdf = buffer_extended_gdf(extended_gdf, obb_gdf, input_raster)
    buffered_gdf_clipped = clip_by_raster_bounding_box(buffered_gdf, input_raster)
    out_image, out_meta = mask_raster_with_all_geometries(input_raster, buffered_gdf)
    if gen_int:
        obb_gdf.to_file(analysis_folder / 'obb.geojson', driver='GeoJSON')
        medial_axis_gdf.to_file(analysis_folder / 'medial_line.geojson', driver='GeoJSON')
        extended_gdf.to_file(analysis_folder / 'coarse_rwy_dir.geojson', driver='GeoJSON')
        buffered_gdf.to_file(analysis_folder / 'coarse_rwy_dir_poly.geojson', driver='GeoJSON')
        buffered_gdf_clipped.to_file(analysis_folder / 'coarse_rwy_dir_poly_clipped.geojson', driver='GeoJSON')
        mask_raster_path = analysis_folder / 'runways_mask.png'
        with rasterio.open(mask_raster_path, 'w', **out_meta) as dest:
            dest.write(out_image)
        print('Raster mask generated with potential areas where runways could be found')
    del obb_gdf, medial_axis_gdf, extended_gdf, buffered_gdf


In [ ]:
import os
from osgeo import gdal
import geopandas as gpd
from pathlib import Path

def make_potential_rwys_horizontal(masks_path, icao):
    masks_path = Path(masks_path)
    src_raster = masks_path / icao / f"runways_mask.png"
    src_vector = masks_path / icao / "coarse_rwy_dir_poly_clipped.geojson"
    src_ds = gdal.Open(str(src_raster))
    gdf = gpd.read_file(src_vector)
    for index, row in gdf.iterrows():
        rotation_angle = abs(int(row['rotation_angle']))
        base_name = src_raster.stem
        rotated_raster_path = masks_path / icao / f"rotated_{base_name}_{index}.tif"
        rotated_vector_path = masks_path / icao / f"horizontal_rwy_cb_{index}.geojson"
        clipped_raster_path = masks_path / icao / f"horizontal_rwy_{index}.tif"
        rotate_image(str(src_raster), str(rotated_raster_path), rotation_angle)
        pivot = get_center(src_ds)
        rotated_gdf = rotate_and_store_gdf(gdf.iloc[[index]], -rotation_angle, pivot)
        rotated_gdf = rotated_gdf.set_crs(gdf.crs)
        rotated_gdf.to_file(rotated_vector_path, driver='GeoJSON')
        gdal_clip_raster_with_vector(str(rotated_raster_path), str(rotated_vector_path), str(clipped_raster_path))
    del src_ds, gdf

In [ ]:
def yolov8_bb_predictions_to_geodf(src_raster, results):
    src = gdal.Open(src_raster)
    ulx, xres, xskew, uly, yskew, yres = src.GetGeoTransform()
    crs_wkt = src.GetProjection()
    ulx = float(ulx); xres = float(xres); xskew = float(xskew)
    uly = float(uly); yskew = float(yskew); yres = float(yres)
    forward_transform = (ulx, xres, xskew, uly, yskew, yres)
    del src
    result = results[0]
    detections_list = []
    names = result.names
    if result.boxes:
        for cls, conf, bbox in zip(result.boxes.cls, result.boxes.conf, result.boxes.xyxy):
            x1, y1, x2, y2 = bbox.tolist()
            lon1, lat1 = gdal.ApplyGeoTransform(forward_transform, float(x1), float(y1))
            lon2, lat2 = gdal.ApplyGeoTransform(forward_transform, float(x2), float(y1))
            lon3, lat3 = gdal.ApplyGeoTransform(forward_transform, float(x2), float(y2))
            lon4, lat4 = gdal.ApplyGeoTransform(forward_transform, float(x1), float(y2))
            latlong_list = [(lon1, lat1), (lon2, lat2), (lon3, lat3), (lon4, lat4), (lon1, lat1)]
            poly = geometry.Polygon([[lon, lat] for lon, lat in latlong_list])
            result_dict = {}
            result_dict['class_id'] = int(cls.item())
            result_dict['class_name'] = names[int(cls.item())]
            result_dict['conf'] = conf.item()
            result_dict['geometry'] = poly
            detections_list.append(result_dict)
    df_preds = pd.DataFrame.from_dict(detections_list)
    gdf_preds = gpd.GeoDataFrame(df_preds)
    gdf_preds.set_crs(crs_wkt, inplace=True)
    return gdf_preds

In [ ]:
from shapely.geometry import LineString
import math

def calculate_runway_direction(polygon):
    oriented_bbox = polygon.minimum_rotated_rectangle
    bbox_coords = list(oriented_bbox.exterior.coords)
    longest_edge = None
    max_length = 0
    for i in range(4):
        p1 = bbox_coords[i]; p2 = bbox_coords[i+1]
        edge = LineString([p1, p2]); length = edge.length
        if length > max_length:
            max_length = length; longest_edge = edge
    p1, p2 = longest_edge.coords
    dx = p2[0] - p1[0]; dy = p2[1] - p1[1]
    azimuth = math.degrees(math.atan2(dx, dy))
    compass_bearing = (azimuth + 360) % 360
    return compass_bearing

In [ ]:
import geopandas as gpd

def process_rwythr_polygons(gdf_unrotated):
    gdf_rwythr = gdf_unrotated[gdf_unrotated['class_name'] == 'rwythr']
    gdf_desig = gdf_unrotated[gdf_unrotated['class_name'] == 'desig']
    rwythr_count = len(gdf_rwythr); desig_count = len(gdf_desig)
    if rwythr_count == 0 and desig_count == 0:
        return None
    elif rwythr_count == 1 and desig_count >= 1:
        gdf_combined = gpd.GeoDataFrame(pd.concat([gdf_rwythr, gdf_desig], ignore_index=True))
    elif rwythr_count == 0 and desig_count >= 2:
        gdf_combined = gdf_desig
    elif rwythr_count == 2 and desig_count == 2:
        gdf_combined = gdf_rwythr
    elif rwythr_count == 2 and desig_count == 0:
        gdf_combined = gdf_rwythr
    elif rwythr_count >= 2 and desig_count >= 2:
        gdf_combined = gdf_rwythr
    else:
        return None
    dissolved_polygons = gdf_combined.dissolve()
    chull_geom = dissolved_polygons.convex_hull
    gdf_unrotated_rwy = gpd.GeoDataFrame(geometry=chull_geom, crs=gdf_unrotated.crs)
    return gdf_unrotated_rwy

In [ ]:
import math
import geomag
from shapely.geometry import LineString

def calculate_runway_id(polygon):
    oriented_bbox = polygon.minimum_rotated_rectangle
    bbox_coords = list(oriented_bbox.exterior.coords)
    longest_edge = None; max_length = 0
    for i in range(4):
        p1 = bbox_coords[i]; p2 = bbox_coords[i+1]
        edge = LineString([p1, p2]); length = edge.length
        if length > max_length:
            max_length = length; longest_edge = edge
    p1, p2 = longest_edge.coords
    dx = p2[0] - p1[0]; dy = p2[1] - p1[1]
    azimuth = math.degrees(math.atan2(dx, dy))
    compass_bearing = (azimuth + 360) % 360
    center_lat = (p1[1] + p2[1]) / 2; center_lon = (p1[0] + p2[0]) / 2
    magnetic_variation = geomag.declination(center_lat, center_lon)
    magnetic_bearing = (compass_bearing + magnetic_variation) % 360
    runway_number_1 = round(magnetic_bearing / 10) % 36
    if runway_number_1 == 0:
        runway_number_1 = 36
    runway_number_2 = (runway_number_1 + 18) % 36
    if runway_number_2 == 0:
        runway_number_2 = 36
    runway_id = f"{runway_number_1:02}-{runway_number_2:02}"
    return runway_id

In [ ]:
def predict_rwy_markings_using_rtdetr(model, masks_path, icao, gen_int=True):
    analysis_folder = Path(masks_path) / icao
    horizontal_rwys_path = glob.glob(os.path.join(analysis_folder, 'horizontal_rwy*.tif'))
    horizontal_rwys_cb_path = glob.glob(os.path.join(analysis_folder, 'horizontal_rwy_cb*.geojson'))
    horizontal_rwys_path.sort(); horizontal_rwys_cb_path.sort()
    pivot = get_pivot(analysis_folder)
    gdf_list = []; gdf_list_rwy = []
    for image_file, cb_path in zip(horizontal_rwys_path, horizontal_rwys_cb_path):
        print(f"Processing {image_file} and {cb_path}")
        rotation_angle = get_unrotate_angle(cb_path)
        results = model(image_file, save=True, save_txt=True, save_conf=True, device=None, show_boxes=True)
        gdf_horizontal = None
        result = results[0]
        if result.boxes:
            gdf_horizontal = yolov8_bb_predictions_to_geodf(image_file, results)
            gdf_horizontal = retain_larger_and_non_overlapping_polygons(gdf_horizontal, 50)
        else:
            continue
        gdf_horizontal = yolov8_bb_predictions_to_geodf(image_file, results)
        if gdf_horizontal is not None and not gdf_horizontal.empty:
            gdf_unrotated = recover_from_rotation(gdf_horizontal, rotation_angle, pivot)
            gdf_list.append(gdf_unrotated)
            gdf_unrotated_rwy = process_rwythr_polygons(gdf_unrotated)
            if gdf_unrotated_rwy is not None:
                gdf_list_rwy.append(gdf_unrotated_rwy)
            if gen_int:
                predicted_geojson = os.path.basename(image_file).split('.')[0] + '.geojson'
                predicted_geojson_path = analysis_folder / predicted_geojson
                gdf_horizontal.to_file(predicted_geojson_path, driver='GeoJSON')
        else:
            print(f"No valid polygons found for {image_file}")
    if gdf_list:
        combined_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))
        combined_gdf.to_file(analysis_folder / "predicted_rwy_markings_bb_rtdetr.geojson", driver='GeoJSON')
    else:
        print("No valid GeoDataFrames to combine.")
    if gdf_list_rwy:
        combined_gdf_rwy = gpd.GeoDataFrame(pd.concat(gdf_list_rwy, ignore_index=True))
        combined_gdf_rwy['runway_id'] = combined_gdf_rwy['geometry'].apply(calculate_runway_id)
        combined_gdf_rwy.to_file(analysis_folder / "predicted_rwy.geojson", driver='GeoJSON')
    else:
        print("No valid 'rwythr' polygons to combine.")

In [ ]:
model_obb = YOLO("models/rwy_obb_v1.pt")

In [ ]:
from ultralytics import RTDETR
yolo_model_path = r"models/rwy_markings_H_v1.pt"
yolo_model_markings_H = RTDETR(yolo_model_path)

**Inference + per-raster timing (workspace-relative; outputs to `reproduce_output_<cat>`)**

In [ ]:
import time

categories = ['single', 'parallel', 'inter', 'mixed', 'complex']
# Imagery is read from rasters/<category>/ (produced by 1_download_dataset.py).
# Zenodo ships JPEG2000 (*.jp2); local GeoTIFFs (*.tif) are also accepted.
RASTER_EXTS = ('*.jp2', '*.tif')
raster_counts = {}
all_times = []
for category in categories:
    in_folder = Path('rasters') / category
    out_folder = f"reproduce_output_{category}"
    rasters_path = sorted(str(p) for ext in RASTER_EXTS for p in in_folder.glob(ext))
    raster_counts[category] = len(rasters_path)
    print(f"{len(rasters_path)} rasters in {in_folder}")
    for input_raster in rasters_path:
        icao = Path(input_raster).stem
        print('\nProcessing', input_raster)
        print('======================================================')
        t0 = time.time()
        try:
            predict_rwy_potential(model_obb, input_raster, out_folder, True)
            make_potential_rwys_horizontal(out_folder, icao)
            predict_rwy_markings_using_rtdetr(yolo_model_markings_H, out_folder, icao)
        except Exception as e:
            print('Issue with', input_raster, str(e))
            all_times.append({'category': category, 'icao': icao, 'inf_time_s': time.time() - t0, 'status': 'failed'})
            continue
        all_times.append({'category': category, 'icao': icao, 'inf_time_s': time.time() - t0, 'status': 'ok'})
        print('Completed processing', input_raster)

Path('metrics').mkdir(exist_ok=True)
df_times = pd.DataFrame(all_times)
df_times.to_csv('metrics/reproduce_output_all_times.csv', index=False)
df_times


In [ ]:
# Merge predicted runways per category into EPSG:4326
ground_truth_gdf = gpd.read_file('ground_truth/all_gt.geojson')
pred_by_cat = {}
for category in categories:
    gdf = gpd.GeoDataFrame()
    for f in glob.glob(f"reproduce_output_{category}/**/predicted_rwy.geojson", recursive=True):
        temp_gdf = gpd.read_file(f)
        temp_gdf = temp_gdf.to_crs(epsg=4326)
        temp_gdf['icao'] = os.path.basename(os.path.dirname(f))
        gdf = gdf._append(temp_gdf, ignore_index=True)
    pred_by_cat[category] = gdf
    print(category, 'pred airports:', gdf['icao'].nunique() if len(gdf) else 0)

In [ ]:
# ICPR Table 3 metric: keep matched pairs (IoU>0.8), per-airport mean, count airports passing
def check_iou(predicted_gdf, ground_truth_gdf):
    results = []
    for icao in predicted_gdf['icao'].unique():
        pred_polygons = predicted_gdf[predicted_gdf['icao'] == icao]
        gt_polygons = ground_truth_gdf[ground_truth_gdf['icao'] == icao]
        for idx_pred, pred_row in pred_polygons.iterrows():
            pred_geom = pred_row.geometry
            for idx_gt, gt_row in gt_polygons.iterrows():
                gt_geom = gt_row.geometry
                intersection = pred_geom.intersection(gt_geom)
                if not intersection.is_empty:
                    union = pred_geom.union(gt_geom)
                    iou = intersection.area / union.area
                    if iou > 0.8:
                        results.append({'icao': icao, 'pred_idx': idx_pred, 'gt_idx': idx_gt, 'iou': iou})
    return results

apt_by_cat = {}
for category in categories:
    df_iou = pd.DataFrame(check_iou(pred_by_cat[category], ground_truth_gdf))
    apt = df_iou.groupby('icao')['iou'].mean().reset_index().rename(columns={'iou': 'apt_iou'})
    apt = apt.merge(df_times[df_times['category'] == category][['icao', 'inf_time_s']], on='icao', how='left')
    apt['category'] = category
    apt_by_cat[category] = apt
apt_all = pd.concat(apt_by_cat.values(), ignore_index=True)
apt_all

**Table 3 — Validation metrics per runway config** (with avg inference time per raster)

In [ ]:
rows = []
for category in categories:
    apt = apt_by_cat[category]
    rows.append({
        'Category': category,
        'Num_Airports': raster_counts[category],
        'IoU_Mean': apt['apt_iou'].mean(),
        'Num_Airports_IoU_80': apt['icao'].nunique(),
        'Avg_Inf_Time_s': df_times[df_times['category'] == category]['inf_time_s'].mean(),
    })
table3 = pd.DataFrame(rows)
Path('metrics').mkdir(exist_ok=True)
table3.to_csv('metrics/reproduce_output_table3.csv', index=False)
print('Average IoU (mean of category means):', round(table3['IoU_Mean'].mean(), 4))
table3


**Multi-seed variability** — quantify run-to-run nondeterminism (mean ± std of category IoU). Reruns N times into seeded folders; supports any subset of categories.

In [ ]:
def run_seeds(seed_categories=('inter',), seeds=(2498, 108, 54)):
    """Re-run the full pipeline once per seed to quantify run-to-run IoU variability.

    Returns a DataFrame with columns: category, seed, Num_Airports, IoU_Mean,
    Num_Airports_IoU_80. Each (seed, category) writes to reproduce_seed{seed}_{category}/.
    Reuses model_obb / yolo_model_markings_H, check_iou and ground_truth_gdf from above.
    """
    import random
    exts = ('*.jp2', '*.tif')
    rows = []
    for seed in seeds:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        for category in seed_categories:
            in_folder = Path('rasters') / category
            out_folder = f"reproduce_seed{seed}_{category}"
            rasters_path = sorted(str(p) for ext in exts for p in in_folder.glob(ext))
            print(f"[seed {seed}] {category}: {len(rasters_path)} rasters -> {out_folder}")
            for input_raster in rasters_path:
                icao = Path(input_raster).stem
                try:
                    predict_rwy_potential(model_obb, input_raster, out_folder, True)
                    make_potential_rwys_horizontal(out_folder, icao)
                    predict_rwy_markings_using_rtdetr(yolo_model_markings_H, out_folder, icao)
                except Exception as e:
                    print('  skip', icao, str(e))
            gdf = gpd.GeoDataFrame()
            for f in glob.glob(f"{out_folder}/**/predicted_rwy.geojson", recursive=True):
                t = gpd.read_file(f).to_crs(epsg=4326)
                t['icao'] = os.path.basename(os.path.dirname(f))
                gdf = gdf._append(t, ignore_index=True)
            df_iou = pd.DataFrame(check_iou(gdf, ground_truth_gdf)) if len(gdf) else pd.DataFrame()
            apt = df_iou.groupby('icao')['iou'].mean() if len(df_iou) else pd.Series(dtype=float)
            rows.append({'category': category, 'seed': seed,
                         'Num_Airports': len(rasters_path),
                         'IoU_Mean': apt.mean(), 'Num_Airports_IoU_80': apt.shape[0]})
    return pd.DataFrame(rows)


In [ ]:
seed_df = run_seeds(seed_categories=('inter',), seeds=(2498, 108, 54))
summary = seed_df.groupby('category')['IoU_Mean'].agg(['mean', 'std', 'min', 'max'])
Path('metrics').mkdir(exist_ok=True)
seed_df.to_csv('metrics/reproduce_seed_variability.csv', index=False)
print(summary)
seed_df


In [ ]:
!python 2_run_inference.py --categories single parallel inter mixed complex